# Notebook 03h — Modelos Lineales con Tratamientos (target: SANGRE)

**Objetivo**: igual que 03c pero usando las fusiones teloméricas en **sangre** como target.  
**Target**: `log1p(Blood_Telomere fusion rate)` — fusiones detectadas en sangre periférica.  
**Features verdes**: firstResponse, treatment_agrupado, Mesenchimal_origin  
**Features azules**: cancer_type, biopsySite, hasSystemicPreTreatment, hasRadiotherapyPreTreatment, treatmentGiven, treatment1  
**Modelos**: Lasso → Ridge → Elastic Net  

**Referencia directa**: 03c usa las mismas features con target = TUMOR (R²=0.085).  
**Pregunta**: ¿Los tratamientos también predicen las fusiones en sangre?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import os
# Idempotente: sube al root del proyecto sólo si estamos dentro de notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f'CWD: {os.getcwd()}')
print('Librerías cargadas OK')

## 1. Carga y unión de datos

In [ ]:
TARGET_SANGRE = 'Blood_Telomere fusion rate (Gt-end/coverage*purity)'

# Base: Patient_ID_IF_mod + cancer_type_code (necesario para merge mesenquimal)
df = pd.read_csv('datos_limpios_modelo_a.csv',
                 usecols=['Patient_ID_IF_mod', 'cancer_type_code'])
print(f'Base: {df.shape}')

# Target de sangre — viene de datos_modelo_final.csv
df_final = pd.read_csv('datos_modelo_final.csv',
                       usecols=['Patient_ID_IF_mod', TARGET_SANGRE])
df = df.merge(df_final, on='Patient_ID_IF_mod', how='inner')
print(f'Tras añadir target sangre: {df.shape}')
print(f'Nulos en target: {df[TARGET_SANGRE].isna().sum()}')

# Features verdes
df_verde = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                         usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment'])
df = df.merge(df_verde, on='Patient_ID_IF_mod', how='left')

# Features azules
df_azul = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'biopsySite', 'hasSystemicPreTreatment',
                                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1'])
df = df.merge(df_azul, on='Patient_ID_IF_mod', how='left')

# Mesenquimal
df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')

print(f'Shape final: {df.shape}')

## 2. Target — distribución y transformación log1p

In [ ]:
print('=== Target original (sangre) ===')
print(df[TARGET_SANGRE].describe())
print(f'Skewness: {df[TARGET_SANGRE].skew():.2f}')
print(f'% ceros exactos: {(df[TARGET_SANGRE] == 0).mean()*100:.1f}%')

y = np.log1p(df[TARGET_SANGRE])
print(f'\nSkewness tras log1p: {y.skew():.2f}')

## 3. Preparación de features

In [ ]:
# Verdes: NaN → Sin_dato
df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

# Azules: NaN → Sin_dato
df['hasSystemicPreTreatment'] = df['hasSystemicPreTreatment'].fillna('Sin_dato')
df['hasRadiotherapyPreTreatment'] = df['hasRadiotherapyPreTreatment'].fillna('Sin_dato')
df['treatmentGiven'] = df['treatmentGiven'].fillna('Sin_dato')

# biopsySite: top 10 + Otro
top10_biopsy = df['biopsySite'].value_counts().head(10).index.tolist()
df['biopsySite_agrupado'] = df['biopsySite'].apply(
    lambda x: x if x in top10_biopsy else ('Sin_dato' if pd.isna(x) else 'Otro')
)

# treatment1: top 10 + Otro
top10_t1 = df['treatment1'].value_counts().head(10).index.tolist()
df['treatment1_agrupado'] = df['treatment1'].apply(
    lambda x: x if x in top10_t1 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

print('Features preparadas OK')

In [ ]:
# Dummies de todas las features (verdes + azules)
cols_features = ['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin',
                 'cancer_type', 'biopsySite_agrupado', 'hasSystemicPreTreatment',
                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1_agrupado']

# cancer_type viene de cancer_type_code — usamos la columna correcta
if 'cancer_type' not in df.columns:
    df['cancer_type'] = df['cancer_type_code']

X = pd.get_dummies(df[cols_features], drop_first=True)
print(f'Shape X: {X.shape} | Shape y: {y.shape}')
print(f'Total features: {X.shape[1]}')

## 4. Split + escalado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

## 5. Función de evaluación

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, feature_names):
    modelo.fit(X_tr, y_tr)
    y_pred_log  = modelo.predict(X_te)
    # Deshacer log1p → métricas en espacio original
    y_pred_orig = np.expm1(y_pred_log)
    y_te_orig   = np.expm1(np.array(y_te))
    r2   = r2_score(y_te_orig, y_pred_orig)
    mae  = mean_absolute_error(y_te_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    cv   = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2').mean()
    sep = '=' * 55
    print(sep)
    print(f'  {nombre}')
    print(sep)
    print(f'  R²  (test, espacio original):  {r2:.4f}')
    print(f'  R²  (CV-5 train):              {cv:.4f}')
    print(f'  MAE  (espacio original):        {mae:.4f}')
    print(f'  RMSE (espacio original):        {rmse:.4f}')
    coefs = pd.Series(modelo.coef_, index=feature_names)
    nz = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    print(f'  Coefs no nulos: {len(nz)} / {len(coefs)}')
    if len(nz) > 0:
        print(nz.head(10).to_string())
    return {'nombre': nombre, 'R2_test': r2, 'R2_cv': cv, 'MAE': mae,
            'RMSE': rmse, 'y_pred': y_pred_orig, 'coefs': coefs}

## 6. Lasso

In [ ]:
res_lasso = evaluar_modelo('Lasso (alpha=0.001)',
    Lasso(alpha=0.001, max_iter=10000, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns)

## 6b. Tratamientos — coeficientes positivos y negativos (Lasso, target: SANGRE)

In [ ]:
coefs = res_lasso['coefs']

prefijos_trat = ('firstResponse_', 'treatment_agrupado_', 'Mesenchimal_origin_',
                 'biopsySite_agrupado_', 'hasSystemicPreTreatment_',
                 'hasRadiotherapyPreTreatment_', 'treatmentGiven_', 'treatment1_agrupado_')
coefs_trat = coefs[[c for c in coefs.index if c.startswith(prefijos_trat)]]
coefs_trat_nz = coefs_trat[coefs_trat != 0].sort_values(ascending=False)

n_total = sum(c.startswith(prefijos_trat) for c in X.columns)
print(f'=== TRATAMIENTOS — Lasso (target: fusiones en SANGRE) ===')
print(f'Variables de tratamiento con coef. no nulo: {len(coefs_trat_nz)} de {n_total}')
print()
print('POSITIVOS (tratamiento asociado a MÁS fusiones en sangre):')
pos = coefs_trat_nz[coefs_trat_nz > 0]
for var, coef in pos.items():
    print(f'  {var:55s}  coef={coef:+.4f}')
print()
print('NEGATIVOS (tratamiento asociado a MENOS fusiones en sangre):')
neg = coefs_trat_nz[coefs_trat_nz < 0]
for var, coef in neg.items():
    print(f'  {var:55s}  coef={coef:+.4f}')

## 7. Ridge

In [ ]:
res_ridge = evaluar_modelo('Ridge (alpha=1.0)',
    Ridge(alpha=1.0, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns)

## 8. Elastic Net

In [ ]:
res_enet = evaluar_modelo('Elastic Net (alpha=0.001, l1_ratio=0.5)',
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns)

## 9. Comparativa

In [ ]:
resultados = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('y_pred', 'coefs')}
    for r in [res_lasso, res_ridge, res_enet]
]).set_index('nombre')

print('=== COMPARATIVA — TRATAMIENTOS, target: SANGRE ===')
print(resultados.round(4).to_string())
print()
print(f'Mejor modelo: {resultados["R2_test"].idxmax()}')
print()
print('--- Referencia 03c (mismas features, target: TUMOR) ---')
print('Lasso:       R2_test=0.0846')
print('Ridge:       R2_test=0.0727')
print('Elastic Net: R2_test=0.0806')

## 10. Gráficos real vs predicho

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, [res_lasso, res_ridge, res_enet]):
    y_pred = np.array(res['y_pred'])
    y_real = np.expm1(np.array(y_test))
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color='darkorange')
    lim = max(y_real.max(), y_pred.max()) if y_pred.max() > 0 else y_real.max()
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1)
    ax.set_xlabel('Real (espacio original, sangre)')
    ax.set_ylabel('Predicho (espacio original, sangre)')
    ax.set_title(f"{res['nombre']}
R²={res['R2_test']:.4f}")
plt.suptitle('Tratamientos (verdes+azules) — TARGET: SANGRE', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/img/img_03h_sangre_tratamientos.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_03h_sangre_tratamientos.png')

## 11. Validacion por Cross-Validation (5-fold) — sobre todos los datos

Hasta ahora el R2=0.051 viene de un solo grupo de test fijo (random_state=42).
Para confirmar que es un resultado solido, aplicamos 5-fold CV sobre todos los pacientes:
cada vuelta usa un 20% distinto como test y el 80% restante para entrenar.
El escalado se hace DENTRO de cada vuelta para no contaminar el test.

Referencia actual (un solo split 80/20, seed=42): Lasso R2=0.051

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

modelos_cv = {
    'Lasso       (alpha=0.001)':         Lasso(alpha=0.001, max_iter=10000, random_state=42),
    'Ridge       (alpha=1.0)':           Ridge(alpha=1.0, random_state=42),
    'Elastic Net (alpha=0.001, l1=0.5)': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42),
}

print('=== CROSS-VALIDATION 5-fold — dataset completo (target: SANGRE) ===')
print(f'  {"Modelo":<40}  {"Media R2":>9}  {"Std":>7}  {"Min":>7}  {"Max":>7}')
print(f'  {"-"*40}  {"-"*9}  {"-"*7}  {"-"*7}  {"-"*7}')

for nombre, modelo in modelos_cv.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', modelo)])
    scores = cross_val_score(pipe, X, y, cv=kf, scoring='r2')
    print(f'  {nombre:<40}  {scores.mean():>9.4f}  {scores.std():>7.4f}  {scores.min():>7.4f}  {scores.max():>7.4f}')

print()
print('Referencia (un solo split seed=42): Lasso=0.0505 | Ridge=0.0347 | ElasticNet=0.0455')